In [160]:
# conda activate anndata

import os
import re
import sys
import pickle
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch
from itertools import combinations
from scipy.stats import rankdata, norm
from statsmodels.stats.multitest import multipletests

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *
from junction2psi import *

In [ ]:
data_source = f"yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_abund_corr"
psi = pd.read_csv(f"data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)
top_qval_mods_df = pd.read_csv(f"data/enrichments/{data_source}.csv")

In [163]:
# Remove modules with < X corr to cell type abundance

mask = ~(top_qval_mods_df['Old_cor'] < .75)
top_qval_mods_df = top_qval_mods_df.loc[mask, :]

In [164]:
# Load single-cell PSI data
sdata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_SJ_counts_PSI_annotated.h5ad")

# Load single-cell gene expression data
adata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_model/yao_2021_ACA_MOp_VISp_STAR_gene_counts_scVI.h5ad")

adata.obs['subclass_label'] = adata.obs['subclass_label'].astype(str).str.replace("/", "_", regex=False).str.replace(" ", "_", regex=False)
sdata.obs['subclass_label'] = sdata.obs['subclass_label'].astype(str).str.replace("/", "_", regex=False).str.replace(" ", "_", regex=False)

# Work with full gene space
adata_raw = adata.raw.to_adata()
adata_raw.X = adata_raw.X.toarray()

In [165]:
intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/psix_annotation/intron_file.tab.gz"
intron_table = read_intron_file(intron_file)

In [166]:
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/gencode.vM35.annotation.gtf"
gtf = gtf_parse(gtf_file)

In [167]:
set(top_qval_mods_df['Cell_type'])

{'Astro',
 'Car3',
 'Endo',
 'L2_3_IT_CTX',
 'L5_6_NP_CTX',
 'L5_PT_CTX',
 'L6_CT_CTX',
 'L6b_CTX',
 'Lamp5',
 'Meis2',
 'Micro-PVM',
 'Oligo',
 'Pvalb',
 'SMC-Peri',
 'Sncg',
 'Sst_Chodl',
 'VLMC',
 'Vip'}

In [168]:
column_order = [
    'Car3', 'L2_3_IT_CTX', 'L4_5_IT_CTX','L5_6_NP_CTX',
    'L5_PT_CTX', 'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX',
    'Pvalb', 'Sst', 'Sst_Chodl', 'Lamp5', 'Sncg', 'Vip', 'Meis2', 
    'Astro', 'Oligo', 'Endo', 'SMC-Peri', 'VLMC', 'Micro-PVM'
]

In [169]:
all_ctypes = [
    'CR', 'Car3', 'L2_3_IT_CTX', 'L2_3_IT_PPP',
    'L4_5_IT_CTX', 'L4_RSP-ACA', 'L5_6_IT_TPE-ENT', 'L5_6_NP_CTX',
    'L5_IT_CTX', 'L5_PT_CTX', 'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX',
    'Pvalb', 'Sst', 'Sst_Chodl', 'Lamp5', 'Sncg', 'Vip', 'Meis2', 
    'Astro', 'Oligo', 'Endo', 'SMC-Peri', 'VLMC', 'Micro-PVM'
    ]

## Format cell type abundance data

In [170]:
top_qval_mods_df.head()

,Cell_type,Pseudobulk_SD,Cor,Old_cor,Pval,Old_pval,Module_genes,Old_module_genes,DE_genes,Module,Old_module,Network,Old_network,ME_path,Old_ME_path,kME_path,Old_kME_path
2,L6_CT_CTX,19.22,0.766757,0.762038,1.568024e-24,2.904979e-27,"Rprm, Foxp2, Ppp1r1b, Ramp3, Ighm, Krt80, Sebo...","Rprm, Ppp1r1b, Foxp2, Ramp3, Ighm, Krt80, Trbc...","Foxp2, Gm10635, Gm30564, Gm35853, Gm40518, Gm4...",bisque3,seashell3,Bicor-None_signum0.24_minSize6_merge_ME_0.95_1...,Bicor-None_signum0.3_minSize6_merge_ME_0.95_18409,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...
3,Vip,20.57,0.797196,0.797196,7.331165e-27,4.585109e-28,"Ap1s2, Vip, Igf1, Prox1, Vstm2l, Cit, Prox1os,...","Ap1s2, Vip, Igf1, Prox1, Vstm2l, Cit, Prox1os,...","9930105H17Rik, Aqp5, Calb2, Chat, Cnmd, Crh, C...",royalblue3,pink4,Bicor-None_signum0.24_minSize10_merge_ME_0.95_...,Bicor-None_signum0.35_minSize6_merge_ME_0.95_1...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...
5,Pvalb,18.79,0.873517,0.873517,8.486837e-27,8.486837e-27,"Pvalb, Rnd2, Kcnc1, Gm13629, Nog, Cox6a2, Kcna...","Pvalb, Rnd2, Kcnc1, Gm13629, Nog, Cox6a2, Kcna...","Adamts15, Adamts8, Adgra3, Akr1c18, Alkal2, C3...",deeppink1,deeppink1,Bicor-None_signum0.266_minSize10_merge_ME_0.95...,Bicor-None_signum0.266_minSize10_merge_ME_0.95...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...
6,L2_3_IT_CTX,20.96,0.880234,0.835110,3.980470e-24,1.028593e-24,"Myh7, Mhrt, Rilpl1, Wfs1, Rgl1, B230216N24Rik,...","Myh7, Mhrt, Rilpl1, Rgl1, Arl15, Gm42864, Gm29...","Agmat, Ccbe1, Evc2, F2rl2, Fst, Gm10421, Gm123...",moccasin,burlywood1,Bicor-None_signum0.24_minSize10_merge_ME_0.95_...,Bicor-None_signum0.438_minSize4_merge_ME_0.95_...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...
7,L6b_CTX,19.91,0.883455,0.845698,2.538891e-26,2.375265e-28,"Nxph4, Cplx3, Tmem40, Phyhipl, Gm50370, Bag1, ...","Nxph4, Cplx3, Tmem40, Phyhipl, Bag1, Prss12, L...","Ccn2, Clic5, Cplx3, Fbxl7, Fzd10os, Galnt10, G...",lightblue2,orangered1,Bicor-None_signum0.724_minSize3_merge_ME_0.95_...,Bicor-None_signum0.3_minSize10_merge_ME_0.95_1...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analy...


In [171]:
ctype_abund = []

for i, row in top_qval_mods_df.iterrows():
    ctype = row['Cell_type']

    mod_df = pd.read_csv(row['Old_ME_path'])
    mod_eig = mod_df.set_index("Sample")[row['Old_module']]
    mod_eig = pd.to_numeric(mod_eig, errors="coerce")
    ctype_abund.append(mod_eig)

In [172]:
ctype_abund_df = pd.concat(ctype_abund, axis=1)
ctype_abund_df.columns = top_qval_mods_df['Cell_type']
ctype_abund_df.head()

Cell_type,L6_CT_CTX,Vip,Pvalb,L2_3_IT_CTX,L6b_CTX,Lamp5,L5_PT_CTX,Sncg,L5_6_NP_CTX,Sst_Chodl,Meis2,Oligo,Car3,VLMC,SMC-Peri,Astro,Endo,Micro-PVM
Sample,,,,,,,,,,,,,,,,,,
Sample1,0.011014,0.039402,-0.105646,0.041033,0.065977,-0.217252,-0.003547,-0.056387,0.129234,-0.071238,0.030502,-0.024297,-0.040620,0.019442,-0.055862,-0.073355,-0.009165,-0.057359
Sample2,0.017395,0.046859,0.068208,-0.017426,-0.036399,-0.083314,-0.010016,0.010431,0.027422,-0.042594,0.072553,-0.060793,-0.056646,0.099773,-0.053389,0.127840,-0.055536,-0.051536
Sample3,0.052864,-0.114202,0.004907,-0.031907,-0.001363,0.059937,0.021052,-0.056851,-0.079066,-0.066356,0.131569,-0.021610,-0.058196,-0.050188,0.039047,0.047864,0.126134,0.059329
Sample4,0.054595,-0.119732,-0.023893,0.048191,-0.006216,0.108865,-0.023398,0.037129,-0.113278,0.231982,-0.062102,0.004649,0.070148,-0.054551,-0.024366,0.041068,-0.019023,0.009389
Sample5,-0.100556,0.159365,-0.049872,-0.066666,0.050518,0.175008,-0.023445,0.229120,-0.079863,-0.046277,-0.034290,0.062759,-0.042377,-0.059672,-0.009831,-0.035451,-0.015797,-0.012038


In [ ]:
ctype_abund_df = ctype_abund_df[column_order]

## Parse GTF (to annotate exons with their genes)

In [ ]:
# Parse GTF attribute column

gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]

## Format PSI data

In [ ]:
psi.head()

,Sample1,Sample2,Sample3,Sample4,Sample5,Sample6,Sample7,Sample8,Sample9,Sample10,...,Sample191,Sample192,Sample193,Sample194,Sample195,Sample196,Sample197,Sample198,Sample199,Sample200
ENSMUSG00000033845_ProteinCoding_1,0.993051,0.993742,0.991429,0.990923,0.992510,0.993995,0.991052,0.993992,0.992033,0.992258,...,0.991705,0.992281,0.991128,0.991474,0.990411,0.992481,0.992344,0.991485,0.991585,0.989809
ENSMUSG00000025903_ProteinCoding_1,0.998125,0.997347,0.996916,0.998937,0.994748,0.993386,0.996764,0.998224,0.998424,0.998615,...,0.998408,0.999413,0.997840,0.996734,0.998484,0.998364,0.998624,0.998346,0.998623,0.999360
ENSMUSG00000025903_ProteinCoding_2,0.986045,0.986886,0.982486,0.980021,0.988957,0.981987,0.981869,0.989045,0.983026,0.986540,...,0.986563,0.983836,0.986367,0.978473,0.981423,0.989912,0.987035,0.986580,0.985086,0.988292
ENSMUSG00000002459_ProteinCoding_1,0.989238,0.989849,0.983940,0.986087,0.983217,0.989082,0.985796,0.986845,0.989053,0.986396,...,0.990198,0.984464,0.987350,0.986844,0.990998,0.974828,0.983984,0.988425,0.989681,0.987894
ENSMUSG00000033793_ProteinCoding_1,0.998637,0.998660,0.998095,0.998607,0.997913,0.998648,0.998156,0.998579,0.998150,0.998553,...,0.998606,0.998411,0.998287,0.998020,0.998059,0.998389,0.998245,0.998480,0.998517,0.998705


In [ ]:
psi.columns = psi.columns.str.replace("-", ".")

# Make sure order of samples matches 
psi = psi[ctype_abund_df.index]

In [ ]:
np.all(psi.columns == ctype_abund_df.index)

True

In [ ]:
# Get PSI data ready to merge on gene IDs
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [ ]:
psi_anno.head()

,gene_name,Sample1,Sample2,Sample3,Sample4,Sample5,Sample6,Sample7,Sample8,Sample9,...,Sample191,Sample192,Sample193,Sample194,Sample195,Sample196,Sample197,Sample198,Sample199,Sample200
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,0.993051,0.993742,0.991429,0.990923,0.992510,0.993995,0.991052,0.993992,0.992033,...,0.991705,0.992281,0.991128,0.991474,0.990411,0.992481,0.992344,0.991485,0.991585,0.989809
ENSMUSG00000025903_ProteinCoding_1,Lypla1,0.998125,0.997347,0.996916,0.998937,0.994748,0.993386,0.996764,0.998224,0.998424,...,0.998408,0.999413,0.997840,0.996734,0.998484,0.998364,0.998624,0.998346,0.998623,0.999360
ENSMUSG00000025903_ProteinCoding_2,Lypla1,0.986045,0.986886,0.982486,0.980021,0.988957,0.981987,0.981869,0.989045,0.983026,...,0.986563,0.983836,0.986367,0.978473,0.981423,0.989912,0.987035,0.986580,0.985086,0.988292
ENSMUSG00000002459_ProteinCoding_1,Rgs20,0.989238,0.989849,0.983940,0.986087,0.983217,0.989082,0.985796,0.986845,0.989053,...,0.990198,0.984464,0.987350,0.986844,0.990998,0.974828,0.983984,0.988425,0.989681,0.987894
ENSMUSG00000033793_ProteinCoding_1,Atp6v1h,0.998637,0.998660,0.998095,0.998607,0.997913,0.998648,0.998156,0.998579,0.998150,...,0.998606,0.998411,0.998287,0.998020,0.998059,0.998389,0.998245,0.998480,0.998517,0.998705


## Get exon coords

In [ ]:
intron_table.head()

,intron
ENSMUSG00000025900_ProteinCoding_1_I1,chr1:4363236-4381492:-
ENSMUSG00000025900_ProteinCoding_1_I2,chr1:4381657-4422132:-
ENSMUSG00000025900_ProteinCoding_1_SE,chr1:4363236-4422132:-
ENSMUSG00000025902_ProteinCoding_1_I1,chr1:4562892-4563322:-
ENSMUSG00000025902_ProteinCoding_1_I2,chr1:4563690-4563994:-


In [ ]:
# Get coordinates for each exon
intron_coords_df = intron_table['intron'].str.split(r"[:\-]", expand=True).iloc[:, :3]
intron_coords_df.columns = ["chr", "intron_first_base", "intron_last_base"]
intron_coords_df.index = intron_table.index
intron_coords_df['exon'] = intron_coords_df.index.str.split("_").str[:3].str.join("_")
intron_coords_df = intron_coords_df[intron_coords_df['exon'].isin(psi_anno.index)] # Subset to exons in PSI data
intron_coords_df['intron_first_base'] = intron_coords_df["intron_first_base"].astype(int)
intron_coords_df['intron_last_base'] = intron_coords_df["intron_last_base"].astype(int)

def safe_exon_coords(g):
    i1 = g.loc[g.index.str.contains("I1$"), "intron_last_base"].values
    i2 = g.loc[g.index.str.contains("I2$"), "intron_first_base"].values
    if len(i1) == 0 or len(i2) == 0:
        return pd.Series({"chr": None, "exon_start": None, "exon_end": None})
    return pd.Series({
        "chr": g["chr"].iloc[0],
        "exon_start": i1[0] + 1,
        "exon_end": i2[0] - 1,
    })


exon_coords_df = intron_coords_df.groupby("exon").apply(safe_exon_coords)
exon_coords_df = exon_coords_df.dropna()  # drop exons with missing coords

/tmp/ipykernel_1088748/3914303081.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exon_coords_df = intron_coords_df.groupby("exon").apply(safe_exon_coords)


In [ ]:
exon_coords_df.head()

,chr,exon_start,exon_end
exon,,,
ENSMUSG00000000028_ProteinCoding_1,chr16,18627482,18627619
ENSMUSG00000000037_ProteinCoding_5,chrX,160038026,160038177
ENSMUSG00000000058_ProteinCoding_1,chr6,17281894,17282081
ENSMUSG00000000078_NMD_1,chr13,5913557,5913624
ENSMUSG00000000085_ProteinCoding_1,chr4,120313482,120313706


## Calc PSI corrs and plot

### Functions

In [ ]:
# =============================================================================
# 1. SPEARMAN CORRELATION + PERMUTATION P-VALUES + FDR + ANALYTICAL CIs
# =============================================================================

def analytical_ci(r, n, ci=0.95):
    """
    Fisher Z-transform based confidence intervals for correlation.
    Exact for Pearson, approximate for Spearman.

    Parameters
    ----------
    r   : array-like, observed correlations
    n   : int, number of samples
    ci  : float, confidence level (default 0.95)

    Returns
    -------
    lower, upper : arrays of CI bounds
    """
    r = np.asarray(r)
    r_clipped = np.clip(r, -0.9999, 0.9999)
    z = np.arctanh(r_clipped)
    se = 1 / np.sqrt(n - 3)
    z_crit = norm.ppf((1 + ci) / 2)
    lower = np.tanh(z - z_crit * se)
    upper = np.tanh(z + z_crit * se)
    return lower, upper


def bootstrap_ci(psi_rank_centered, ct_ranks, n_boot=1000, ci=95, seed=42):
    """
    Bootstrap confidence intervals for Spearman correlation.
    Only call this for specific exons you intend to plot — slow for all exons.

    Parameters
    ----------
    psi_rank_centered : np.array, shape (n_exons, n_samples)
    ct_ranks          : np.array, shape (n_samples,)
    n_boot            : int, number of bootstrap iterations
    ci                : float, confidence level (default 95)
    seed              : int, random seed

    Returns
    -------
    lower, upper : arrays of CI bounds, shape (n_exons,)
    """
    np.random.seed(seed)
    n_samples = psi_rank_centered.shape[1]

    boot_indices = np.random.choice(n_samples, size=(n_boot, n_samples), replace=True)

    ct_boot_all = ct_ranks[boot_indices]                                        # (n_boot, n_samples)
    ct_boot_centered = ct_boot_all - ct_boot_all.mean(axis=1, keepdims=True)   # (n_boot, n_samples)
    ct_boot_norm = np.sqrt((ct_boot_centered**2).sum(axis=1))                  # (n_boot,)

    boot_corrs = np.zeros((n_boot, psi_rank_centered.shape[0]))
    for i in range(n_boot):
        idx = boot_indices[i]
        psi_boot = psi_rank_centered[:, idx]
        psi_boot_norm = np.sqrt(np.nansum(psi_boot**2, axis=1, keepdims=True))
        boot_corrs[i, :] = (psi_boot @ ct_boot_centered[i]) / (psi_boot_norm.squeeze() * ct_boot_norm[i])

    lower = np.percentile(boot_corrs, (100 - ci) / 2, axis=0)
    upper = np.percentile(boot_corrs, 100 - (100 - ci) / 2, axis=0)
    return lower, upper


def spearman_permutation_test(psi_numeric, ctype_abund_df, psi_anno,
                               n_perms=1000, ci=0.95, seed=42):
    """
    Compute Spearman correlations between each PSI event and each cell type,
    with permutation-based p-values, BH-FDR correction, and analytical CIs.
    Also returns permuted correlations for use in permutation-based
    correlation difference test.
 
    Parameters
    ----------
    psi_numeric     : pd.DataFrame, shape (n_exons, n_samples)
    ctype_abund_df  : pd.DataFrame, shape (n_samples, n_celltypes)
    psi_anno        : pd.DataFrame with 'gene_name' column
    n_perms         : int, number of permutations
    ci              : float, confidence level for analytical CIs
    seed            : int, random seed
 
    Returns
    -------
    psi_corr_df       : Spearman r values
    psi_pval_df       : permutation p-values
    psi_fdr_df        : BH-FDR corrected p-values (within each cell type)
    psi_ci_lower_df   : analytical CI lower bounds
    psi_ci_upper_df   : analytical CI upper bounds
    perm_corr_results : dict {ct: (n_exons, n_perms) array} of permuted correlations
    psi_rank_centered : (n_exons, n_samples) centered rank matrix (for bootstrap CIs)
    psi_rank_norm     : (n_exons, 1) rank norms (for bootstrap CIs)
    ct_ranks_dict     : dict {ct: (n_samples,) array} of ranked ct values (for bootstrap CIs)
    """
    np.random.seed(seed)
 
    n_samples = psi_numeric.shape[1]
 
    # --- precompute PSI ranks once ---
    psi_array = psi_numeric.values  # (n_exons, n_samples)
    psi_rank_matrix = np.apply_along_axis(
        lambda x: rankdata(np.where(np.isnan(x), np.nan, x), method='average'),
        axis=1,
        arr=psi_array
    )  # (n_exons, n_samples)
 
    # center PSI ranks once
    psi_rank_centered = psi_rank_matrix - np.nanmean(psi_rank_matrix, axis=1, keepdims=True)
    psi_rank_norm = np.sqrt(np.nansum(psi_rank_centered ** 2, axis=1, keepdims=True))  # (n_exons, 1)
 
    # --- precompute all permutation indices ---
    perm_indices = np.array([np.random.permutation(n_samples) for _ in range(n_perms)])
    # shape: (n_perms, n_samples)
 
    corr_results = {}
    pval_results = {}
    perm_corr_results = {}
    ct_ranks_dict = {}
 
    for ct in ctype_abund_df.columns:
        ct_vals = ctype_abund_df[ct].values
        ct_ranks = rankdata(ct_vals, method='average')
        ct_ranks_dict[ct] = ct_ranks
 
        # --- observed correlation ---
        ct_centered = ct_ranks - ct_ranks.mean()
        ct_norm = np.sqrt((ct_centered ** 2).sum())
        observed = (psi_rank_centered @ ct_centered) / (psi_rank_norm.squeeze() * ct_norm)
        # shape: (n_exons,)
 
        # --- all permutations at once ---
        ct_perm_ranks = ct_ranks[perm_indices]          # (n_perms, n_samples)
        ct_perm_centered = ct_perm_ranks - ct_perm_ranks.mean(axis=1, keepdims=True)
        ct_perm_norm = np.sqrt((ct_perm_centered ** 2).sum(axis=1))  # (n_perms,)
 
        # (n_exons, n_samples) @ (n_samples, n_perms) = (n_exons, n_perms)
        num = psi_rank_centered @ ct_perm_centered.T
        denom = psi_rank_norm * ct_perm_norm[np.newaxis, :]  # (n_exons, n_perms)
        perm_corrs = num / denom                             # (n_exons, n_perms)
 
        # --- two-tailed permutation p-values ---
        pvals = (np.abs(perm_corrs) >= np.abs(observed[:, np.newaxis])).mean(axis=1)
 
        corr_results[ct] = observed
        pval_results[ct] = pvals
        perm_corr_results[ct] = perm_corrs  # store for correlation difference test
 
        print(f"Done: {ct}")
 
    # --- assemble output dataframes ---
    psi_corr_df = pd.DataFrame(corr_results, index=psi_numeric.index)
    psi_corr_df.insert(0, 'Gene', psi_anno['gene_name'])
 
    psi_pval_df = pd.DataFrame(pval_results, index=psi_numeric.index)
    psi_pval_df.insert(0, 'Gene', psi_anno['gene_name'])
 
    # --- BH-FDR correction within each cell type ---
    psi_fdr_df = psi_pval_df.copy()
    for ct in ctype_abund_df.columns:
        pvals = psi_pval_df[ct].values
        mask = ~np.isnan(pvals)
        fdr = np.full(len(pvals), np.nan)
        _, fdr[mask], _, _ = multipletests(pvals[mask], method='fdr_bh')
        psi_fdr_df[ct] = fdr
 
    # --- analytical CIs ---
    n = psi_numeric.shape[1]
    psi_ci_lower_df = psi_corr_df.copy()
    psi_ci_upper_df = psi_corr_df.copy()
    for ct in ctype_abund_df.columns:
        lower, upper = analytical_ci(psi_corr_df[ct].values, n, ci=ci)
        psi_ci_lower_df[ct] = lower
        psi_ci_upper_df[ct] = upper
 
    return (psi_corr_df, psi_pval_df, psi_fdr_df, psi_ci_lower_df, psi_ci_upper_df,
            perm_corr_results, psi_rank_centered, psi_rank_norm, ct_ranks_dict)


# =============================================================================
# 2. PERMUTATION-BASED CORRELATION DIFFERENCE TEST
# =============================================================================

def compare_all_ct_pairs(psi_corr_df, perm_corr_results, psi_anno, ct_cols):
    """
    Permutation-based test for difference between dependent correlations.
    Reuses permuted correlations from spearman_permutation_test — no extra
    computation needed.

    For each pair (ct1, ct2), the null distribution of r(PSI, ct1) - r(PSI, ct2)
    is computed from the permuted correlations. This avoids the analytical
    Steiger formula and its det_R <= 0 edge cases.

    Parameters
    ----------
    psi_corr_df       : pd.DataFrame, observed correlations (no Gene column)
    perm_corr_results : dict {ct: (n_exons, n_perms) array}
    psi_anno          : pd.DataFrame with 'gene_name' column
    ct_cols           : list of cell type column names

    Returns
    -------
    dict keyed by (ct1, ct2) tuples, values are DataFrames with columns:
        Gene, r_ct1, r_ct2, r_diff, pval, fdr
    where r_diff = r(PSI, ct1) - r(PSI, ct2), positive means ct1 > ct2
    """
    all_pvals = []
    all_keys = []
    all_masks = []
    results = {}

    for ct1, ct2 in combinations(ct_cols, 2):
        observed_ct1 = psi_corr_df[ct1].values   # (n_exons,)
        observed_ct2 = psi_corr_df[ct2].values   # (n_exons,)

        perm_ct1 = perm_corr_results[ct1]         # (n_exons, n_perms)
        perm_ct2 = perm_corr_results[ct2]         # (n_exons, n_perms)

        observed_diff = observed_ct1 - observed_ct2           # (n_exons,)
        perm_diff = perm_ct1 - perm_ct2                       # (n_exons, n_perms)

      # one-tailed p-value in the observed direction
        pvals = np.where(
            observed_diff >= 0,
            (perm_diff >= observed_diff[:, np.newaxis]).mean(axis=1),  # p(ct1 > ct2)
            (perm_diff <= observed_diff[:, np.newaxis]).mean(axis=1)   # p(ct1 < ct2)
        )

        mask = ~np.isnan(pvals)
        results[(ct1, ct2)] = {
            'observed_diff': observed_diff,
            'pvals': pvals, 
            'r_ct1': observed_ct1,
            'r_ct2': observed_ct2
        }
        #             'mask': mask,
        
        # collect valid (non-NaN) pvals for FDR correction
        # we concatenate across all pairs so FDR is corrected globally
        all_pvals.append(pvals[mask])
        all_keys.append((ct1, ct2))
        all_masks.append(mask)

     # --- FDR correct all pairs together, one correction per direction ---
    all_pvals_concat = np.concatenate(all_pvals)
    _, all_fdr_concat, _, _ = multipletests(all_pvals_concat, method='fdr_bh')

    # split FDR back into per-pair results
    idx = 0
    final_results = {}
    for (ct1, ct2), mask in zip(all_keys, all_masks):
        n_valid = mask.sum()

        fdr = np.full(len(mask), np.nan)
        fdr[mask] = all_fdr_concat[idx:idx + n_valid]
        idx += n_valid

        d = results[(ct1, ct2)]
        final_results[(ct1, ct2)] = pd.DataFrame({
            'Gene': psi_anno['gene_name'].values,
            f'r_{ct1}': d['r_ct1'],
            f'r_{ct2}': d['r_ct2'],
            'r_diff': d['observed_diff'],       # positive = ct1 > ct2
            'pval': d['pvals'],  
            'fdr': fdr       
        }, index=psi_corr_df.index)

    return final_results

# =============================================================================
# 3. FIND CELL-TYPE-SPECIFIC EXONS
# =============================================================================

def find_specific_exons_per_ct_basic(psi_fdr_df, psi_corr_df, fdr_thresh=0.05):
    """
    For each cell type, find exons where FDR < fdr_thresh.

    Returns
    -------
    dict of {ct: DataFrame}
    """
    ct_cols = [c for c in psi_fdr_df.columns if c != 'Gene']
    results = {}
    for target_ct in ct_cols:
        mask = psi_fdr_df[target_ct] < fdr_thresh
        n_total = mask.sum()
        print(f"{target_ct}: {n_total} significant exons")
        results[target_ct] = psi_corr_df[mask].sort_values(target_ct, ascending=False)
    return results


def find_specific_exons_per_ct(steiger_results, psi_fdr_df, psi_corr_df, ct_hierarchy, fdr_thresh=0.05, ascending=False):
    """
    For each cell type, find exons where:
    1. FDR < fdr_thresh for that cell type
    2. Correlation difference is significantly greater (or lesser if ascending)
       than all non-child cell types by permutation-based test (r_diff direction + FDR < fdr_thresh). Children defined in ct_hierarchy are excluded.

    Parameters
    ----------
    steiger_results : dict from compare_all_ct_pairs
    psi_fdr_df      : FDR DataFrame (no Gene column)
    psi_corr_df     : correlation DataFrame (with Gene column)
    ct_hierarchy    : dict {ct: [children to skip]}
    fdr_thresh      : float
    ascending       : bool, if True find exons where target has LOWEST correlation

    Returns
    -------
    dict of {ct: DataFrame}
    """
    ct_cols = [c for c in psi_fdr_df.columns if c != 'Gene']
    results = {}

    for target_ct in ct_cols:
        children_to_skip = ct_hierarchy.get(target_ct, [])
        other_cts = [ct for ct in ct_cols if ct != target_ct and ct not in children_to_skip]

        # condition 1: significant in target cell type
        sig_mask = psi_fdr_df[target_ct] < fdr_thresh

        # condition 2: significantly greater (or lesser) than all non-child cell types
        greater_mask = pd.Series(True, index=psi_fdr_df.index)

        for other_ct in other_cts:
            
            if (target_ct, other_ct) in steiger_results:
                df = steiger_results[(target_ct, other_ct)]
                if ascending:
                    ct_greater = (df['r_diff'] < 0) & (df['fdr'] < fdr_thresh)
                else:
                    ct_greater = (df['r_diff'] > 0) & (df['fdr'] < fdr_thresh)
            elif (other_ct, target_ct) in steiger_results:
                df = steiger_results[(other_ct, target_ct)]
                if ascending:
                    ct_greater = (df['r_diff'] > 0) & (df['fdr'] < fdr_thresh)
                else:
                    ct_greater = (df['r_diff'] < 0) & (df['fdr'] < fdr_thresh)
            else:
                print(f"Warning: no result found for ({target_ct}, {other_ct})")
                ct_greater = pd.Series(False, index=psi_fdr_df.index)

            greater_mask = greater_mask & ct_greater

        mask = sig_mask & greater_mask
        n_total = sig_mask.sum()
        n_specific = mask.sum()
        print(f"{target_ct}: {n_specific} specific exons ({n_total} significant total)")

        results[target_ct] = psi_corr_df[mask].sort_values(target_ct, ascending=ascending)

    return results


# =============================================================================
# 4. COMBINE RESULTS
# =============================================================================

def combine_results(ctype_specific_exons_basic, ctype_specific_exons_strict,
                    psi_corr_df, psi_fdr_df, steiger_results, ct_hierarchy, exon_coords_df):
    """
    For each cell type, combine basic (FDR only) and strict (FDR + permutation
    difference test) results into a single table.

    Steiger columns (r_diff, fdr) are filled only for specific exons,
    NaN for non-specific exons.

    Returns
    -------
    dict of {ct: DataFrame}
    """
    ct_cols = [c for c in psi_corr_df.columns if c != 'Gene']
    combined = {}

    for target_ct in ct_cols:
        # start from all FDR-significant exons
        all_sig = ctype_specific_exons_basic.get(target_ct, pd.DataFrame()).copy()
        if len(all_sig) == 0:
            continue

        # mark which exons passed strict criterion in either direction
        strict_true = set(ctype_specific_exons_strict['True'].get(target_ct, pd.DataFrame()).index)
        strict_false = set(ctype_specific_exons_strict['False'].get(target_ct, pd.DataFrame()).index)
        all_sig['is_specific'] = all_sig.index.isin(strict_true | strict_false)
        all_sig['specific_direction'] = np.where(
            all_sig.index.isin(strict_false), 'highest',
            np.where(all_sig.index.isin(strict_true), 'lowest', np.nan)
        )

        # add r and fdr for target cell type
        all_sig['r'] = psi_corr_df.loc[all_sig.index, target_ct]
        all_sig['fdr'] = psi_fdr_df.loc[all_sig.index, target_ct]
        all_sig['exon_start'] = exon_coords_df.loc[all_sig.index, 'exon_start']
        all_sig['exon_end'] = exon_coords_df.loc[all_sig.index, 'exon_end']
        all_sig['exon_len'] = all_sig['exon_end'] - all_sig['exon_start'] + 1
        all_sig['chr'] = exon_coords_df.loc[all_sig.index, 'chr']

        # add r_diff and fdr for non-child cell types — only for specific exons
        children_to_skip = ct_hierarchy.get(target_ct, [])
        other_cts = [ct for ct in ct_cols if ct != target_ct and ct not in children_to_skip]

        specific_idx = all_sig[all_sig['is_specific']].index

        for other_ct in other_cts:
            # initialize all as NaN
            all_sig[f'r_diff_{other_ct}'] = np.nan
            all_sig[f'fdr_diff_{other_ct}'] = np.nan  # renamed

            if len(specific_idx) == 0:
                continue

            if (target_ct, other_ct) in steiger_results:
                sdf = steiger_results[(target_ct, other_ct)]
                r_diff_vals = sdf.loc[specific_idx, 'r_diff'].values
                fdr_vals = sdf.loc[specific_idx, 'fdr'].values
            elif (other_ct, target_ct) in steiger_results:
                sdf = steiger_results[(other_ct, target_ct)]
                r_diff_vals = sdf.loc[specific_idx, 'r_diff'].values * -1  # flip sign
                fdr_vals = sdf.loc[specific_idx, 'fdr'].values             # fdr stays the same
            else:
                continue

            all_sig.loc[specific_idx, f'r_diff_{other_ct}'] = r_diff_vals
            all_sig.loc[specific_idx, f'fdr_diff_{other_ct}'] = fdr_vals

        combined[target_ct] = all_sig.sort_values('r', ascending=False)

    return combined

# =============================================================================
# 5. PLOTTING
# =============================================================================

def _safe(name):
    # simple filename sanitizer
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

def _barplot_colors(ctypes, working_ctype):
    colors = ["tab:blue"] * len(ctypes)
    idx = pd.Index(ctypes).get_indexer([working_ctype])[0]
    colors[idx] = "tab:red"
    return colors

def _flatten_1d(x):
    from scipy import sparse
    if sparse.issparse(x):   # AnnData/sparse safe
        return x.toarray().ravel()
    return np.asarray(x, dtype=float).ravel()

def plot_barplot_by_cell_type(means, errs, ctypes, working_ctype, title, ylabel,
                              counts=None, show=True):
    means = np.asarray(means)
    errs = np.asarray(errs) * 2.0

    colors = _barplot_colors(ctypes, working_ctype)
    x = np.arange(len(ctypes))

    if counts is not None:
        # Two stacked panels: main barplot on top, cell counts on bottom
        fig, (ax, ax_n) = plt.subplots(
            2, 1,
            gridspec_kw={'height_ratios': [3, 1]},
        )
        # Top: main barplot
        ax.bar(x, means, yerr=errs, capsize=3, color=colors)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)

        # Bottom: cell counts in gray
        ax_n.bar(x, counts, color="gray")
        ax_n.set_ylabel("# cells")
        ax_n.set_xticks(x)
        ax_n.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)

        fig.tight_layout()
        fig.subplots_adjust(hspace=0.6)  # more space since both have rotated labels
    else:
        fig, ax = plt.subplots()
        ax.bar(x, means, yerr=errs, capsize=3, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        fig.tight_layout()

    if show:
        plt.show()
    return fig

def plot_ME_vs_PSI(mod_eig, exon_psi, title, show=True):
    fig, ax = plt.subplots()
    ax.scatter(mod_eig, exon_psi, s=18)
    ax.set_xlabel("Module eigengene")
    ax.set_ylabel("Exon PSI")
    ax.set_title(title)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

def violin_with_points(values_by_ct, ctypes, focus=None,
                       base_fc='tab:blue', highlight_fc='tab:red',
                       ylabel="Value", title=None,
                       jitter=0.08, point_size=8, point_alpha=0.35,
                       max_points_per_ct=None, seed=0, violin_alpha=0.5,
                       show=False):
    rng = np.random.default_rng(seed)
    pos = np.arange(1, len(ctypes)+1)
    fig, ax = plt.subplots()
    vp = ax.violinplot(values_by_ct, positions=pos, showmeans=True, showextrema=True)
    # color violins (no outlines)
    for i, b in enumerate(vp['bodies']):
        col = highlight_fc if (focus is not None and ctypes[i] == focus) else base_fc
        b.set_facecolor(col)
        b.set_edgecolor('none')
        b.set_alpha(violin_alpha)
    for k in ('cmeans','cmins','cmaxes','cbars','cmedians'):
        if k in vp: vp[k].set_visible(False)
    # jitter points
    for i, v in enumerate(values_by_ct, start=1):
        v = np.asarray(v, float)
        v = v[np.isfinite(v)]
        if v.size == 0: continue
        if max_points_per_ct and v.size > max_points_per_ct:
            v = v[rng.choice(v.size, size=max_points_per_ct, replace=False)]
        x = i + (rng.random(v.size) - 0.5) * 2 * jitter
        col = highlight_fc if (focus is not None and ctypes[i-1] == focus) else base_fc
        ax.scatter(x, v, s=point_size, alpha=point_alpha, color=col, edgecolors='none', rasterized=True)
    ax.set_xticks(pos)
    ax.set_xticklabels(ctypes, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel)
    if title: ax.set_title(title)
    fig.tight_layout()
    if show:
        plt.show()
    return fig

### Run

In [ ]:
ct_dict = {ct: ct for ct in column_order}

for ct in column_order:
    ct_dict[ct] = []
    
ct_hierarchy = ct_dict

In [178]:
psi_numeric = psi_anno.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

# --- correlations + permutation p-values + FDR + CIs ---
(psi_corr_df, psi_pval_df, psi_fdr_df, psi_ci_lower_df, psi_ci_upper_df,
 perm_corr_results, psi_rank_centered, psi_rank_norm, ct_ranks_dict) = spearman_permutation_test(
    psi_numeric,
    ctype_abund_df,
    psi_anno,
    n_perms=10000,
    ci=0.95
)
print("")

# --- permutation-based correlation difference test ---
steiger_results = compare_all_ct_pairs(
    psi_corr_df.drop(columns='Gene'),
    perm_corr_results,
    psi_anno,
    ctype_abund_df.columns.tolist()
)
# --- find specific exons ---
ctype_specific_exons_basic = find_specific_exons_per_ct_basic(
    psi_fdr_df.drop(columns='Gene'),
    psi_corr_df,
    fdr_thresh=0.05
)
print("")

ctype_specific_exons_strict = {}
for ascending in (True, False):
    print("ascending:", ascending)
    ctype_specific_exons_strict[str(ascending)] = find_specific_exons_per_ct(
        steiger_results,
        psi_fdr_df.drop(columns='Gene'),
        psi_corr_df,
        ct_hierarchy,
        fdr_thresh=0.05,
        ascending=ascending
    )
    print("")

# --- combine results ---
combined_results = combine_results(
    ctype_specific_exons_basic,
    ctype_specific_exons_strict,
    psi_corr_df,
    psi_fdr_df.drop(columns='Gene'),
    steiger_results,
    ct_hierarchy,
    exon_coords_df
)

Done: L6_CT_CTX
Done: Vip
Done: Pvalb
Done: L2_3_IT_CTX
Done: L6b_CTX
Done: Lamp5
Done: L5_PT_CTX
Done: Sncg
Done: L5_6_NP_CTX
Done: Sst_Chodl
Done: Meis2
Done: Oligo
Done: Car3
Done: VLMC
Done: SMC-Peri
Done: Astro
Done: Endo
Done: Micro-PVM

L6_CT_CTX: 65 significant exons
Vip: 54 significant exons
Pvalb: 81 significant exons
L2_3_IT_CTX: 80 significant exons
L6b_CTX: 342 significant exons
Lamp5: 15 significant exons
L5_PT_CTX: 9 significant exons
Sncg: 36 significant exons
L5_6_NP_CTX: 3 significant exons
Sst_Chodl: 49 significant exons
Meis2: 415 significant exons
Oligo: 1001 significant exons
Car3: 31 significant exons
VLMC: 1202 significant exons
SMC-Peri: 1054 significant exons
Astro: 793 significant exons
Endo: 1580 significant exons
Micro-PVM: 1241 significant exons

ascending: True
L6_CT_CTX: 0 specific exons (65 significant total)
Vip: 0 specific exons (54 significant total)
Pvalb: 7 specific exons (81 significant total)
L2_3_IT_CTX: 1 specific exons (80 significant total)
L

In [179]:
# Save ctype exons
for ctype, df in combined_results.items():
    print(ctype)
    df.to_csv(f"data/ctype_exons/{_safe(ctype)}_exons.csv")

L6_CT_CTX
Vip
Pvalb
L2_3_IT_CTX
L6b_CTX
Lamp5
L5_PT_CTX
Sncg
L5_6_NP_CTX
Sst_Chodl
Meis2
Oligo
Car3
VLMC
SMC-Peri
Astro
Endo
Micro-PVM


### Plot distro of corrs

In [ ]:
fdr_thresh = 0.05
ct_cols = [c for c in psi_fdr_df.columns if c != 'Gene']

n_cols = 2
n_rows = int(np.ceil(len(ct_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 2.5 * n_rows), dpi=200, sharex=True)  # <-- shared x-axis

axes_flat = axes.flatten()

for i, ct in enumerate(ct_cols):
    ax = axes_flat[i]
    observed = psi_corr_df[ct].values
    null = perm_corr_results[ct].flatten()
    sig_mask = (psi_fdr_df[ct] < fdr_thresh).values

    ax.tick_params(labelbottom=True)
    ax.hist(null, bins=100, color='lightgrey', density=True, label='Null (permuted)')
    ax.hist(observed, bins=100, color='steelblue', alpha=0.6, density=True, label='Observed (all)')
    ax.hist(observed[sig_mask], bins=100, color='salmon', alpha=0.8, density=True, 
            label=f'Significant (FDR < {fdr_thresh})')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(ct)
    ax.set_ylabel('Density')
    # if i >= len(ct_cols) - n_cols:  # only bottom row gets x label
    ax.set_xlabel('Spearman r')
    ax.legend(fontsize=7)

# hide any unused axes
for j in range(len(ct_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('figures/correlation_distributions.pdf', dpi=200)
plt.close()

### Plot cell type-specific exons

In [182]:
gene_exon_df = psi_corr_df['Gene']

In [185]:
# --- plot strict specific exons ---

# for ascending in (True, False):

ascending = True 

ctype_exons = ctype_specific_exons_strict[str(ascending)]

for target_ct, exon_df in ctype_exons.items():
    if len(exon_df) == 0:
        print(f"Skipping {target_ct} — no specific exons")
        continue
        
    outdir = f"figures/pseudobulk/{data_source}/{target_ct}"
    os.makedirs(outdir, exist_ok=True)
    
    print(f"Plotting {len(exon_df)} exons for {target_ct}")
    
    mask = top_qval_mods_df['Cell_type'] == target_ct 
    row = top_qval_mods_df[mask]
    mod_df = pd.read_csv(row['Old_ME_path'].item())
    mod_eig_df = mod_df.set_index("Sample")[row['Old_module'].item()]
    mod_eig = pd.to_numeric(mod_eig_df, errors="coerce")

    for idx, row in exon_df.iterrows():
        gene = row['Gene']
        exon_suffix = "_".join(idx.split("_")[1:])

        # Get mean expression and variance of exon/gene in each cell type
        exon_mask = sdata.var_names.isin([idx])
        sdata_sub = sdata[:, exon_mask].copy()
        gene_mask = adata_raw.var_names.isin([gene_exon_df.loc[idx]])
        adata_sub = adata_raw[:, gene_mask].copy()
        
        psi_vals_by_ct = []
        expr_vals_by_ct = []
        mean_psi_by_ct = [] 
        mean_psi_se_by_ct = []
        mean_expr_by_ct = []
        mean_expr_se_by_ct = []
        n_by_ct = []  # NEW: collect cell counts per type

        for ct in all_ctypes:
            cell_mask = adata_sub.obs['subclass_label'] == ct
            n = np.sum(cell_mask)
            n_by_ct.append(int(n))  # NEW

            psi_per_cell = _flatten_1d(sdata_sub.X[cell_mask, :])
            expr_per_cell = _flatten_1d(adata_sub.X[cell_mask, :])

            psi_vals_by_ct.append(psi_per_cell)
            expr_vals_by_ct.append(np.log1p(expr_per_cell))

            mean_psi_by_ct.append(np.mean(psi_per_cell))
            mean_psi_se_by_ct.append(np.sqrt(np.var(psi_per_cell) / n))

            mean_expr = np.mean(adata_raw.X[cell_mask, :]) 
            mean_expr_by_ct.append(np.mean(expr_per_cell))
            mean_expr_se_by_ct.append(np.sqrt(np.var(expr_per_cell) / n) / mean_expr)
    
        corr = round(psi_corr_df.loc[idx, target_ct], 3)
        gene = gene_exon_df.loc[idx]
        pdf_path = f"{outdir}/{_safe(target_ct)}_{_safe(gene)}_{_safe(exon_suffix)}_ascending{ascending}.pdf"
        
        # exon coordinates
        start_coord = int(exon_coords_df.loc[idx]['exon_start'])
        end_coord   = int(exon_coords_df.loc[idx]['exon_end'])
        exon_len    = end_coord - start_coord + 1
        exon_info = f"{exon_len} bp ({start_coord}-{end_coord})"
                 
        with PdfPages(pdf_path) as pdf:
            fig = plot_barplot_by_cell_type(mean_psi_by_ct, mean_psi_se_by_ct, 
                                            all_ctypes, target_ct,
                                            title=f"PSI for {gene} {exon_suffix} exon\n{exon_info}",
                                            ylabel="Mean PSI",
                                            counts=n_by_ct,  # NEW
                                            show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

            fig = plot_barplot_by_cell_type(mean_expr_by_ct, mean_expr_se_by_ct, 
                                            all_ctypes, target_ct,
                                            title=f"Gene expression for {gene}",
                                            ylabel="Mean expression (normalized)",
                                            counts=n_by_ct,  # NEW
                                            show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

            fig = plot_ME_vs_PSI(mod_eig.values, psi_numeric.loc[idx].values,
                                 title=f"{target_ct} module eigengene vs. PSI for {gene} {exon_suffix} exon\nCorr: {corr}\n{exon_info}",
                                 show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

            fig = violin_with_points(psi_vals_by_ct, all_ctypes, focus=target_ct,
                                     ylabel="PSI",
                                     title=f"PSI distribution for {gene} {exon_suffix} exon\n{exon_info}",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()
            
            fig = violin_with_points(expr_vals_by_ct, all_ctypes, focus=target_ct,
                                     ylabel="Expression (log2)",
                                     title=f"Count distribution for {gene}",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

        print("Saved", gene, exon)
        print("")

Skipping L6_CT_CTX — no specific exons
Skipping Vip — no specific exons
Plotting 7 exons for Pvalb
Saved Snap91 ProteinCoding_2

Saved Kcnc2 ProteinCoding_2

Saved Oxr1 ProteinCoding_2

Saved Pnkd ProteinCoding_2

Saved Kcnip2 ProteinCoding_2

Saved Kcnip2 ProteinCoding_2

Saved Map7d2 ProteinCoding_2

Plotting 1 exons for L2_3_IT_CTX
Saved Ppp3ca ProteinCoding_2

Skipping L6b_CTX — no specific exons
Skipping Lamp5 — no specific exons
Skipping L5_PT_CTX — no specific exons
Plotting 1 exons for Sncg
Saved Kcnip1 ProteinCoding_2

Skipping L5_6_NP_CTX — no specific exons
Plotting 2 exons for Sst_Chodl
Saved Ahi1 ProteinCoding_2

Saved Crkl ProteinCoding_2

Plotting 28 exons for Meis2
Saved Pcbp3 ProteinCoding_2

Saved Pcbp3 ProteinCoding_2

Saved Dync1i1 ProteinCoding_2

Saved Ypel3 ProteinCoding_2

Saved Zfp385b ProteinCoding_2

Saved Epb41l3 ProteinCoding_2

Saved Dbn1 ProteinCoding_2

Saved Pcbp3 ProteinCoding_2

Saved Epha5 ProteinCoding_2

Saved Camk2b ProteinCoding_2

Saved Tafazzin

In [ ]:
c